## Necessary initialization

In [5]:
from pathlib import Path
import sys, shutil

import pandas as pd
from loguru import logger
from datetime import datetime, timedelta

# Relative imports
from collector import YahooCollectorUS, Run


# Append dump_bin to sys.path
CUR_DIR = Path().resolve()
sys.path.append(str(CUR_DIR.parent.parent))

from dump_bin import DumpDataAll, DumpDataUpdate, DumpDataFix


# Target database
qlib_dir = Path().resolve().parent.parent.parent.joinpath("qlib_data_US")
qlib_dir

WindowsPath('D:/Projects/Investing/QuantitativeML_Investing/Database/qlib_data_US')

## Rewrite YahooCollectorUS.get_instrument_list
To pull only selected symbols

In [6]:
# Rewrite YahooCollectorUS.get_instrument_list
tickers_sp500 = pd.read_csv(qlib_dir.joinpath("instruments/sp500_only_new.txt"), sep='\t', header=None, names=['symbol', 'start_date', 'end_date'])
tickers_sp400 = pd.read_csv(qlib_dir.joinpath("instruments/sp400_only_new.txt"), sep='\t', header=None, names=['symbol', 'start_date', 'end_date'])


tickers = (
    ['^GSPC']  # '^MID', '^SP600'
    + tickers_sp500['symbol'].tolist()
    + tickers_sp400['symbol'].tolist()
)
tickers = [ticker.replace('.', '-') for ticker in tickers]


def f(self):

    logger.info("get US stock symbols......")

    symbols = tickers

    logger.info(f"get {len(symbols)} symbols.")
    return symbols


YahooCollectorUS.get_instrument_list = f


## RUN

In [7]:
r = Run(region="US", interval="1d")

### Download data

In [8]:
r.download_data(
    max_collector_count=2,
    delay=0.5,
    start='1996-01-01',
    end=(datetime.today()).strftime('%Y-%m-%d'),
)

2026-02-14 20:30:57.219 | INFO     | __main__:f:16 - get US stock symbols......
2026-02-14 20:30:57.221 | INFO     | __main__:f:20 - get 904 symbols.
2026-02-14 20:30:57.721 | INFO     | data_collector.base:collector_data:202 - start collector data......
2026-02-14 20:30:57.722 | INFO     | data_collector.base:collector_data:207 - getting data: 1
100%|██████████| 904/904 [47:38<00:00,  3.16s/it]


[]


2026-02-14 21:18:35.777 | INFO     | data_collector.base:_collector:195 - error symbol nums: 0
2026-02-14 21:18:35.779 | INFO     | data_collector.base:_collector:196 - current get symbol nums: 904
2026-02-14 21:18:35.781 | INFO     | data_collector.base:collector_data:209 - 1 finish.
2026-02-14 21:18:35.782 | INFO     | data_collector.base:collector_data:216 - total 904, error: 0


### Normalize data

In [9]:
r.normalize_data()

2026-02-14 21:18:35.817 | INFO     | data_collector.utils:get_calendar_list:67 - get calendar list: US_ALL......


                          open         high          low        close  \
symbol date                                                             
^GSPC  1942-01-02     8.890000     8.890000     8.890000     8.890000   
       1942-01-05     9.090000     9.090000     9.090000     9.090000   
       1942-01-06     9.050000     9.050000     9.050000     9.050000   
       1942-01-07     9.000000     9.000000     9.000000     9.000000   
       1942-01-08     8.880000     8.880000     8.880000     8.880000   
...                        ...          ...          ...          ...   
       2026-02-09  6917.259766  6980.100098  6905.870117  6964.819824   
       2026-02-10  6974.490234  6986.830078  6937.529785  6941.810059   
       2026-02-11  6976.479980  6993.479980  6911.970215  6941.470215   
       2026-02-12  6957.540039  6973.220215  6824.040039  6832.759766   
       2026-02-13  6834.270020  6881.959961  6794.549805  6836.169922   

                       volume     adjclose  
symbo

2026-02-14 21:18:42.229 | INFO     | data_collector.utils:get_calendar_list:93 - end of get calendar list: US_ALL.
2026-02-14 21:18:42.231 | INFO     | data_collector.base:normalize:313 - normalize data......
100%|██████████| 904/904 [03:32<00:00,  4.25it/s]


## DUMP DATA

In [12]:
_dump = DumpDataAll(
    data_path=Path().resolve().joinpath("source"),
    qlib_dir=qlib_dir,
    exclude_fields="symbol,date",
    max_workers=4,
)
_dump.dump()

2026-02-14 22:11:03.738 | INFO     | dump_bin:_get_all_date:307 - start get all date......
100%|██████████| 904/904 [00:31<00:00, 28.71it/s]
2026-02-14 22:11:35.233 | INFO     | dump_bin:_get_all_date:326 - end of get all date.

2026-02-14 22:11:35.235 | INFO     | dump_bin:_dump_calendars:329 - start dump calendars......
2026-02-14 22:11:35.315 | INFO     | dump_bin:_dump_calendars:332 - end of calendars dump.

2026-02-14 22:11:35.316 | INFO     | dump_bin:_dump_instruments:335 - start dump instruments......
2026-02-14 22:11:35.322 | INFO     | dump_bin:_dump_instruments:337 - end of instruments dump.

2026-02-14 22:11:35.323 | INFO     | dump_bin:_dump_features:340 - start dump features......
100%|██████████| 904/904 [00:53<00:00, 16.88it/s]
2026-02-14 22:12:28.891 | INFO     | dump_bin:_dump_features:347 - end of features dump.



## Clear

In [13]:
shutil.rmtree(Path().resolve().joinpath("source"))
shutil.rmtree(Path().resolve().joinpath("normalize"))